## 7.3 ToolStrategy使用详解

ToolStrategy通过 工具调用 （Tool Calling/Function Calling）实现结构化输出，所以LangChain会在
消息列表末尾 追加一条ToolMessage ，让整个链路完整。但实际上没有实际的工具执行，这是一条伪
消息。

ToolStrategy适用于任何支持工具调用的现代模型。

ToolStrategy的配置包含三个主要参数：

```python
ToolStrategy(
  self,
  schema: type[SchemaT] | UnionType | dict[str, Any],
  *,
  tool_message_content: str | None = None,
  handle_errors: bool | str | type[Exception] | tuple[type[Exception], ...] | Callable[[Exception], str] = True
)
```

* schema（必需参数）：与提供商策略的schema参数功能一致，支持 Pydantic模型 、TypedDict 、 JSON Schema 、 数据类(@dataclass) ，同时还支持联合类型 Union[类型1, 类型2] （允许模型根据输入内容选择最匹配的数据结构）。

* tool_message_content（可选参数）：用于自定义生成结构化输出时，会话历史中记录的提示信
息。默认使用展示输出数据的标准响应语句。

* handle_errors（可选参数）：用于指定数据校验失败时的重试策略， 默认值为True 。

### 7.3.1 结构化输出：schema参数

下面演示四种Schema进行结构化输出代码实现。

提前说明：因为涉及到不同的Schema在不同模型供应商下表现的支持力度不同（上一章有说明，这里提供了两个模型供应商，大家自己选择）

使用OpenAI平台的模型：

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = ChatOpenAI(
    model="gpt-6-luna",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
)

输出模式1：Pydantic类型

Pydantic类型的Schema支持数据验证，是优先推荐使用的方式。

In [2]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint
# 定义工具@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息
    
    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费"""
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件
    
    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
        "请分析指定客户的情况："
        "1. 先搜索客户数据库了解最新情况 "
        "2. 如果是VIP客户，则发送感谢邮件 "
        "3. 基于搜索结果生成结构化分析报告 "
        "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
    ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)
# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
# "messages": [{"role": "user","content": "请分析客户李四"}]
# "messages": [{"role": "user","content": "请分析客户王五"}]
# "messages": [{"role": "user","content": "今天天气如何"}]
})
# 处理结果
#rprint(result)
if "structured_response" in result:
    analysis = result["structured_response"]
    rprint(analysis)

CustomerAnalysis(
    customer_name='张三',
    customer_tier='VIP客户',
    recent_activity='最近购买日期：2026-01-15；累计消费：$15,000',
    spending_level='高',
    send_email=True
)

In [3]:
rprint(result)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='eb127f53-63c5-49cd-b699-b47507e9604b'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"query":"张三"}',
                    'call_id': 'call_zcDkuLDlCqhrAW2VgistKxiY',
                    'name': 'search_customer_database',
                    'type': 'function_call',
                    'id': 'fc_0b97d7ab40e8ba49016aba6d26899c87d2853a49e6286134a6',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'create_time': 1790602533.901404,
                        'turn_id': '01a0e83a-58c2-7b9b-bc0c-b0c9e802a89f'
                    },
                    'metadata': {'turn_id': '01a0e83a-58c2-7b9b-bc0c-b0c9e802a89f'}
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0b97d7ab40e8ba49016aba6d25b67887d2aa0849880ad180e2',
                'created_at': 1790602533.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0b97d7ab40e8ba49016aba6d25b67887d2aa0849880ad180e2',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_zcDkuLDlCqhrAW2VgistKxiY',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4696,
                'output_tokens': 20,
                'total_tokens': 4716,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='00304031-ac55-430e-897e-1241797e7bb5',
            tool_call_id='call_zcDkuLDlCqhrAW2VgistKxiY'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"customer":"张三"}',
                    'call_id': 'call_w5Vu6EOBI5Lmuq4o0oxEgdXS',
                    'name': 'send_email',
                    'type': 'function_call',
                    'id': 'fc_0b97d7ab40e8ba49016aba6d2a7a1887d29cb4ce8bd0703591',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'create_time': 1790602537.839688,
                        'turn_id': '01a0e83a-66c8-7d07-8cfd-b78dadc8af9b'
                    },
                    'metadata': {'turn_id': '01a0e83a-66c8-7d07-8cfd-b78dadc8af9b'}
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0b97d7ab40e8ba49016aba6d29982887d29ad357c8028c7143',
                'created_at': 1790602537.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0b97d7ab40e8ba49016aba6d29982887d29ad357c8028c7143',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_w5Vu6EOBI5Lmuq4o0oxEgdXS',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
       

输出模式5: 多schema联合模式

ToolStrategy允许指定多个类型“ Union[类型1, 类型2] ”这种写法，LLM能够根据输入文本的内容，智能地选择 最合适的一个 数据模型（Schema）来生成结构化输出，**但是最终会只有一种类型输出**。

适用于根据不同输入内容，生成不同的结构化输出的场景，但是底层工具转换结构化输出只会转换成一
种结构化类型输出。

In [4]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")
class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")
agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventInfo]
    )
)
response = agent.invoke({
    "messages": [
    HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
    ]
})
for msg in response["messages"]:
    msg.pretty_print()
print(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
================================== Ai Message ==================================

[{'arguments': '{"name":"小明","email":"shkstart@atguigu.com","phone":"12345678912"}', 'call_id': 'call_LWpxynLGJDUmlCdTSYzog1UB', 'name': 'ContactInfo', 'type': 'function_call', 'id': 'fc_1790604291720_3euzodvmq47', 'status': 'completed'}]
Tool Calls:
  ContactInfo (call_LWpxynLGJDUmlCdTSYzog1UB)
 Call ID: call_LWpxynLGJDUmlCdTSYzog1UB
  Args:
    name: 小明
    email: shkstart@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'
name='小明' email='shkstart@atguigu.com' phone='12345678912'


### 7.3.2 自定义工具消息：tool_message_content参数
> 如果采用ToolStrategy策略处理结构化输出时，LangChain会在消息列表末尾追加一条Tool_message，让整个链路完整。但实际上没有实际的工具执行，这是一条伪消息。

我们可以通过ToolStrategy的 tool_message_content 参数定制其消息内容，将指定的内容写入对话历
史的提示信息，这样做的好处如下：
1. 在最终用户可见的对话流中，使用 更自然的消息 替代原始数据。
2. 用简短的确认信息替代可能很长的数据块， 减少token消耗 。
举例：

当不设置 tool_message_content时，模型收到的 ToolMessage里就包含了像 {'name': '张三', 'email':'zhangsan@email.com'... ...} 这样的具体数据。当设置了tool_message_content时，模型收到的ToolMessage只是一个预定义的确认信息，如`格式化输出成功！`。

这种方式节省了上下文窗口的令牌消耗，并且让对话流对最终用户更友好。

说明： 无论 tool_message_content如何设置，成功提取的结构化数据最终都会正确存入result["structured_response"] 返回，自定义消息仅影响对话历史中的一条记录。

In [5]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")
class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")
agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema=Union[ContactInfo, EventInfo],
        tool_message_content="格式化输出成功!"
    )
)
response = agent.invoke({
    "messages": [
    HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
    ]
})
for msg in response["messages"]:
    msg.pretty_print()
print(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
================================== Ai Message ==================================

[{'arguments': '{"email":"shkstart@atguigu.com","name":"小明","phone":"12345678912"}', 'call_id': 'call_FIYqpWl4ITEhDOguMCiJl8wH', 'name': 'ContactInfo', 'type': 'function_call', 'id': 'fc_0046a547627fda45016aba7bd18bf887d29d45385b104b1a44', 'status': 'completed', 'internal_chat_message_metadata_passthrough': {'create_time': 1790606288.595059, 'turn_id': '01a0e873-a49f-7f28-9311-99b2f1772fbb'}, 'metadata': {'turn_id': '01a0e873-a49f-7f28-9311-99b2f1772fbb'}}]
Tool Calls:
  ContactInfo (call_FIYqpWl4ITEhDOguMCiJl8wH)
 Call ID: call_FIYqpWl4ITEhDOguMCiJl8wH
  Args:
    email: shkstart@atguigu.com
    name: 小明
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

格式化输出成功!
name='小明' email='shkstart@atguig

In [6]:
rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='d9b61277-2909-4ac1-95d3-be54b07b5c6b'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"email":"shkstart@atguigu.com","name":"小明","phone":"12345678912"}',
                    'call_id': 'call_FIYqpWl4ITEhDOguMCiJl8wH',
                    'name': 'ContactInfo',
                    'type': 'function_call',
                    'id': 'fc_0046a547627fda45016aba7bd18bf887d29d45385b104b1a44',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'create_time': 1790606288.595059,
                        'turn_id': '01a0e873-a49f-7f28-9311-99b2f1772fbb'
                    },
                    'metadata': {'turn_id': '01a0e873-a49f-7f28-9311-99b2f1772fbb'}
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0046a547627fda45016aba7bd05f6887d2b89014447ac1572f',
                'created_at': 1790606288.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0046a547627fda45016aba7bd05f6887d2b89014447ac1572f',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'email': 'shkstart@atguigu.com', 'name': '小明', 'phone': '12345678912'},
                    'id': 'call_FIYqpWl4ITEhDOguMCiJl8wH',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4507,
                'output_tokens': 36,
                'total_tokens': 4543,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='格式化输出成功!',
            name='ContactInfo',
            id='b98f0bc8-6dda-459e-874b-eb958a42d254',
            tool_call_id='call_FIYqpWl4ITEhDOguMCiJl8wH'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}